Linen usage prediction for the next 30 days using Prophet model
> Using Azure ML SDK v2 (MLClient) with Prophet forecasting model

In [ ]:

# Install SDK v2 and Prophet (run once in the notebook if needed)
#!pip install --quiet azure-ai-ml azure-identity prophet mlflow

In [5]:
%pip show azure-ai-ml

Name: azure-ai-ml
Version: 1.28.1
Summary: Microsoft Azure Machine Learning Client Library for Python
Home-page: https://github.com/Azure/azure-sdk-for-python
Author: Microsoft Corporation
Author-email: azuresdkengsysadmins@microsoft.com
License: MIT License
Location: /anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages
Requires: azure-common, azure-core, azure-mgmt-core, azure-monitor-opentelemetry, azure-storage-blob, azure-storage-file-datalake, azure-storage-file-share, colorama, isodate, jsonschema, marshmallow, msrest, pydash, pyjwt, pyyaml, six, strictyaml, tqdm, typing-extensions
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [6]:
# Connect using MLClient (SDK v2)
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient, Input
from azure.ai.ml.entities import AmlCompute

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
except Exception:
    credential = InteractiveBrowserCredential()

# MLClient will read configuration from ./config.json or env vars if present
ml_client = MLClient.from_config(credential=credential)
print('MLClient initialized for subscription/workspace')

Found the config file in: /config.json


MLClient initialized for subscription/workspace


In [7]:

import pandas as pd
df = pd.read_csv('./LinenData.csv')
print(df.columns.tolist())
print(df.head())

['Date', 'AdmCount', 'DosaCount', 'DayOfWeekNum', 'IsWeekend', 'BlanketUsage']
         Date  AdmCount  DosaCount  DayOfWeekNum  IsWeekend  BlanketUsage
0  28/11/2025         4          0             5          0            20
1  27/11/2025         4          0             4          0             0
2  26/11/2025        10          3             3          0             0
3  25/11/2025        11          3             2          0             0
4  24/11/2025         6          1             1          0             0


In [8]:

target_column_name = 'BlanketUsage'
time_column_name = 'Date'
forecast_horizon = 30
df[time_column_name] = pd.to_datetime(df[time_column_name], format='%d/%m/%Y')
df = df.sort_values(time_column_name).reset_index(drop=True)
print(df.dtypes)
print('Rows:', len(df))

Date            datetime64[ns]
AdmCount                 int64
DosaCount                int64
DayOfWeekNum             int64
IsWeekend                int64
BlanketUsage             int64
dtype: object
Rows: 348


In [9]:
# Create or get a compute target via MLClient (AmlCompute entity)
compute_name = 'aml-cluster-dev'
existing = {c.name: c for c in ml_client.compute.list()}
if compute_name in existing:
    print('Found compute:', compute_name)
else:
    print('Creating compute:', compute_name)
    compute = AmlCompute(name=compute_name, size='STANDARD_DS11_V2', min_instances=0, max_instances=4)
    ml_client.compute.begin_create_or_update(compute).result()
    print('Compute created')

Found compute: aml-cluster-dev


In [10]:
# Configure MLflow local tracking (change URI to server if you have one)
import mlflow
mlflow.set_tracking_uri('file:./mlruns')
mlflow.set_experiment('linen-forecast-experiment')
print('MLflow tracking URI:', mlflow.get_tracking_uri())

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.1.1) and mlflow-skinny (2.22.1) are different. This may lead to unexpected behavior. Please install the same version of both packages.
  mlflow.mismatch._check_version_mismatch()
2026/01/09 16:36:06 INFO mlflow.tracking.fluent: Experiment with name 'linen-forecast-experiment' does not exist. Creating a new experiment.


MLflow tracking URI: file:./mlruns


In [11]:
import os
print("Files in current directory:", os.listdir('.'))
if os.path.exists('./data'):
    print("Files in data directory:", os.listdir('./data'))
else:
    print("data/ directory does NOT exist - creating it now")
    os.makedirs('./data', exist_ok=True)
    
    # Copy LinenData.csv to data folder
    import shutil
    shutil.copy('./LinenData.csv', './data/LinenData.csv')
    
    # Create MLTable file with correct format
    mltable_content = """type: mltable
paths:
  - file: ./LinenData.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
"""
    with open('./data/MLTable', 'w') as f:
        f.write(mltable_content)
    
    print("Created data/ folder with MLTable and LinenData.csv")

# Always recreate MLTable with correct encoding if it exists
if os.path.exists('./data/MLTable'):
    mltable_content = """type: mltable
paths:
  - file: ./LinenData.csv
transformations:
  - read_delimited:
      delimiter: ','
      encoding: 'utf8'
      header: all_files_same_headers
"""
    with open('./data/MLTable', 'w') as f:
        f.write(mltable_content)
    print("Updated MLTable file with correct encoding")

Files in current directory: ['.amlignore', '.amlignore.amltmp', '.git', '.gitignore', '.ipynb_aml_checkpoints', 'Deploy-add storage.ipynb', 'LinenData.csv', 'linen_blanket_prediction.ipynb', 'mlruns', 'Setup', 'Training linen Prophet.ipynb', 'training linen prophet.ipynb.amltmp', 'Training linen.ipynb', 'training linen.ipynb.amltmp']
data/ directory does NOT exist - creating it now
Created data/ folder with MLTable and LinenData.csv
Updated MLTable file with correct encoding


# Train Prophet Model for Blanket Usage

Now we'll train a Prophet model for the blanket usage time series and log everything to Azure ML using MLflow.

In [12]:
# Install Prophet if not already installed
import subprocess
import sys

try:
    import prophet
    print("Prophet is already installed")
except ImportError:
    print("Installing Prophet...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "prophet"])
    print("Prophet installed successfully")
    import prophet  # Import after installation

# Verify Prophet is available
from prophet import Prophet
print("Prophet version:", prophet.__version__)

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Importing plotly failed. Interactive plots will not work.


Prophet is already installed
Prophet version: 1.2.1


In [14]:
# Train Prophet model for blanket usage and log locally via MLflow
from prophet import Prophet
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import json
import os

# Use local MLflow tracking (already configured in cell 8)
# Note: Prophet models work better with local tracking
mlflow.set_experiment('linen-forecast-prophet')

# Start MLflow run for the training process
with mlflow.start_run(run_name="prophet_blanket_forecast") as run:
    mlflow.log_param('model_type', 'Prophet')
    mlflow.log_param('forecast_horizon', forecast_horizon)
    mlflow.log_param('time_column', time_column_name)
    mlflow.log_param('target_column', target_column_name)
    mlflow.log_param('linen_type', 'Blanket')
    
    print(f"\n{'='*60}")
    print(f"Training Prophet model for Blanket Usage")
    print(f"{'='*60}")
    
    # Prepare data in Prophet format (ds, y)
    prophet_df = df[[time_column_name, target_column_name]].copy()
    prophet_df.columns = ['ds', 'y']
    prophet_df = prophet_df.sort_values('ds').reset_index(drop=True)
    
    print(f"Training data size: {len(prophet_df)} days")
    print(f"Date range: {prophet_df['ds'].min()} to {prophet_df['ds'].max()}")
    print(f"Target statistics:")
    print(f"  Mean: {prophet_df['y'].mean():.2f}")
    print(f"  Std: {prophet_df['y'].std():.2f}")
    print(f"  Min: {prophet_df['y'].min():.2f}")
    print(f"  Max: {prophet_df['y'].max():.2f}")
    
    # Create and train Prophet model
    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True,
        changepoint_prior_scale=0.05,
        seasonality_prior_scale=10.0
    )
    
    # Add additional regressors if available
    if 'AdmCount' in df.columns:
        model.add_regressor('AdmCount')
        prophet_df['AdmCount'] = df['AdmCount'].values
        print("Added AdmCount as regressor")
    
    if 'DosaCount' in df.columns:
        model.add_regressor('DosaCount')
        prophet_df['DosaCount'] = df['DosaCount'].values
        print("Added DosaCount as regressor")
    
    if 'IsWeekend' in df.columns:
        model.add_regressor('IsWeekend')
        prophet_df['IsWeekend'] = df['IsWeekend'].values
        print("Added IsWeekend as regressor")
    
    model.fit(prophet_df)
    
    # Make future dataframe for forecasting
    future = model.make_future_dataframe(periods=forecast_horizon, freq='D')
    
    # Add regressor values for future dates (using mean for simplicity)
    if 'AdmCount' in df.columns:
        future['AdmCount'] = df['AdmCount'].mean()
    if 'DosaCount' in df.columns:
        future['DosaCount'] = df['DosaCount'].mean()
    if 'IsWeekend' in df.columns:
        # Calculate IsWeekend based on day of week
        future['IsWeekend'] = future['ds'].dt.dayofweek.isin([5, 6]).astype(int)
    
    forecast = model.predict(future)
    
    # Calculate metrics on historical data
    historical_pred = forecast[forecast['ds'].isin(prophet_df['ds'])]
    historical_actual = prophet_df.merge(historical_pred[['ds', 'yhat']], on='ds')
    
    rmse = np.sqrt(mean_squared_error(historical_actual['y'], historical_actual['yhat']))
    mae = mean_absolute_error(historical_actual['y'], historical_actual['yhat'])
    
    # Handle MAPE calculation when y can be zero
    non_zero_mask = historical_actual['y'] != 0
    if non_zero_mask.sum() > 0:
        mape = np.mean(np.abs((historical_actual.loc[non_zero_mask, 'y'] - historical_actual.loc[non_zero_mask, 'yhat']) / historical_actual.loc[non_zero_mask, 'y'])) * 100
    else:
        mape = 0
    
    print(f"\nModel Metrics:")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    print(f"  MAPE: {mape:.2f}%")
    
    # Log metrics to MLflow
    mlflow.log_metric('rmse', rmse)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('mape', mape)
    
    # Create visualization
    fig = model.plot(forecast)
    plt.title('Prophet Forecast for Blanket Usage')
    plt.xlabel('Date')
    plt.ylabel('Blanket Usage')
    plt.tight_layout()
    plot_path = 'forecast_plot_blanket.png'
    plt.savefig(plot_path)
    mlflow.log_artifact(plot_path)
    plt.close()
    
    # Create components plot
    fig2 = model.plot_components(forecast)
    plt.tight_layout()
    components_path = 'components_plot_blanket.png'
    plt.savefig(components_path)
    mlflow.log_artifact(components_path)
    plt.close()
    
    # Save forecast to CSV
    forecast_csv_path = 'forecast_blanket.csv'
    future_forecast = forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(forecast_horizon).copy()
    future_forecast.to_csv(forecast_csv_path, index=False)
    mlflow.log_artifact(forecast_csv_path)
    
    print(f"\nFuture forecast for Blanket Usage (next {forecast_horizon} days):")
    print(future_forecast.to_string())
    
    # Log model using Prophet-specific logger (local tracking only)
    try:
        # Create an input example for the model signature
        input_example = future.head(5)  # Use first 5 rows of future dataframe
        
        mlflow.prophet.log_model(
            model, 
            "prophet_model_blanket",
            input_example=input_example
        )
        print("\nProphet model logged successfully to local MLflow (with signature)")
    except Exception as e:
        print(f"\nNote: Could not log Prophet model: {e}")
        print("Model training completed successfully, but model logging was skipped")
    
    print(f"\n{'='*60}")
    print(f"Training completed successfully!")
    print(f"MLflow Run ID: {run.info.run_id}")
    print(f"View results locally: mlflow ui (then open http://localhost:5000)")
    print(f"{'='*60}")

16:45:24 - cmdstanpy - INFO - Chain [1] start processing
16:45:24 - cmdstanpy - INFO - Chain [1] done processing



Training Prophet model for Blanket Usage
Training data size: 348 days
Date range: 2024-12-03 00:00:00 to 2025-11-28 00:00:00
Target statistics:
  Mean: 4.48
  Std: 8.08
  Min: 0.00
  Max: 40.00
Added AdmCount as regressor
Added DosaCount as regressor
Added IsWeekend as regressor

Model Metrics:
  RMSE: 3.76
  MAE: 2.04
  MAPE: 19.21%


2026/01/09 16:45:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Future forecast for Blanket Usage (next 30 days):
            ds       yhat  yhat_lower  yhat_upper
348 2025-11-29  -0.694295   -4.979119    4.316239
349 2025-11-30  -0.522877   -5.254237    4.703904
350 2025-12-01  -0.503685   -5.434546    4.204592
351 2025-12-02  11.112336    6.579989   15.650859
352 2025-12-03  -0.876403   -5.731428    3.480392
353 2025-12-04   0.171711   -4.899381    5.136642
354 2025-12-05  18.455206   13.752024   23.511603
355 2025-12-06   0.875534   -3.794901    5.555132
356 2025-12-07   1.135833   -3.933409    5.695339
357 2025-12-08   1.216270   -3.460478    5.759433
358 2025-12-09  12.864950    8.284820   17.801617
359 2025-12-10   0.879960   -3.727900    6.107304
360 2025-12-11   1.903220   -2.281989    6.934262
361 2025-12-12  20.134170   14.996380   24.812097
362 2025-12-13   2.475751   -2.144926    7.367583
363 2025-12-14   2.633110   -1.995489    7.530257
364 2025-12-15   2.588901   -2.226152    7.067201
365 2025-12-16  14.094108    8.860097   18.274234

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/prophet/serialize.py:172: FutureWarning: The behavior of Timestamp.utcfromtimestamp is deprecated, in a future version will return a timezone-aware Timestamp wi


Prophet model logged successfully to local MLflow (with signature)

Training completed successfully!
MLflow Run ID: c8aace905fb040abb9e91dde26a3d345
View results locally: mlflow ui (then open http://localhost:5000)
